# EKB End-to-End Orchestrator Verification

This notebook verifies the full `KBIngestionPipeline` workflow, including:
1. **Upload**: Local document to GCS Landing Zone with custom metadata.
2. **Classification**: Step 01 (DLP) and Step 02 (Gemini) classification and routing.
3. **RAG Ingestion**: Step 03 semantic chunking and Step 04 BQML vectorization.

# EKB End-to-End Orchestrator Verification

This notebook verifies the full `KBIngestionPipeline` workflow, including:
1. **Upload**: Local document to GCS Landing Zone with custom metadata.
2. **Classification**: Step 01 (DLP) and Step 02 (Gemini) classification and routing.
3. **RAG Ingestion**: Step 03 semantic chunking and Step 04 BQML vectorization.

## 1. Setup and Initialization

In [ ]:
import sys
import os
import uuid
from google.cloud import storage, bigquery

# Ensure project root is in path
sys.path.append("../..")

from pipelines.enterprise_knowledge_base.orchestrator import KBIngestionPipeline, OrchestratorRunRequest

# Configuration
PROJECT_ID = "ag-core-dev-fdx7"
os.environ["PROJECT_ID"] = PROJECT_ID
LANDING_BUCKET = f"{PROJECT_ID}-kb-landing-zone"
LOCAL_FILE = "SoW - Innovation.pdf"

orchestrator = KBIngestionPipeline(PROJECT_ID)
storage_client = storage.Client()

print(f"Orchestrator initialized for project: {PROJECT_ID}")

2026-04-27 19:05:03.546 | INFO     | pipelines.enterprise_knowledge_base.rag_ingestion.pipeline:__init__:39 - Initialized RAGIngestion | CHUNK_SIZE: 1000 | OVERLAP: 100


Orchestrator initialized for project: ag-core-dev-fdx7


## 2. Upload Document to Landing Zone

We upload the local file to the landing zone and inject custom metadata (X-Goog-Meta) that the pipeline uses for classification.

In [9]:
def ingest_local_file(local_path: str, metadata: dict) -> str:
    client = storage.Client()
    bucket_name = "ag-core-dev-fdx7-kb-landing-zone"
    filename = os.path.basename(local_path)
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(filename)
    blob.metadata = metadata
    blob.upload_from_filename(local_path)
    return f"gs://{bucket_name}/{filename}"

In [ ]:
# 1. Ingest
local_path = "../../SoW - Innovation.pdf"
metadata = {
    "project": "orchestration-test",
    "domain": "it",
    "trust-level": "archived",
    "uploader": "emmanuel.amador@endava.com"
}

if os.path.exists(local_path):
    landing_uri = ingest_local_file(local_path, metadata)
    print(f"Ingested: {landing_uri}")

2026-04-27 19:11:41.958 | INFO     | pipelines.enterprise_knowledge_base.rag_ingestion.pipeline:__init__:39 - Initialized RAGIngestion | CHUNK_SIZE: 1000 | OVERLAP: 100


Ingested: gs://ag-core-dev-fdx7-kb-landing-zone/SoW - Innovation.pdf


In [16]:
landing_uri

'gs://ag-core-dev-fdx7-kb-landing-zone/SoW - Innovation.pdf'

## 3. Run End-to-End Pipeline

This triggers the entire workflow: Classification -> Routing -> Chunking -> Vectorization.

In [ ]:
request = OrchestratorRunRequest(gcs_uri=landing_uri)
resp = orchestrator.run(request)

2026-04-27 19:11:42.859 | INFO     | pipelines.enterprise_knowledge_base.orchestrator:run:39 - Triggering KB Ingestion Pipeline for: gs://ag-core-dev-fdx7-kb-landing-zone/SoW - Innovation.pdf
2026-04-27 19:11:42.860 | INFO     | pipelines.enterprise_knowledge_base.orchestrator:run:42 - Step 1: Running Document Classification...
2026-04-27 19:11:42.860 | INFO     | pipelines.enterprise_knowledge_base.document_classification.pipeline:run:55 - Starting pipeline orchestration for: gs://ag-core-dev-fdx7-kb-landing-zone/SoW - Innovation.pdf
2026-04-27 19:11:42.860 | DEBUG    | pipelines.enterprise_knowledge_base.document_classification.pipeline:_get_blob_metadata:191 - Extracting metadata for: gs://ag-core-dev-fdx7-kb-landing-zone/SoW - Innovation.pdf
2026-04-27 19:11:42.861 | INFO     | pipelines.enterprise_knowledge_base.document_classification.gcs_service.service:get_blob_metadata:31 - Extracting detailed GCS metadata for: gs://ag-core-dev-fdx7-kb-landing-zone/SoW - Innovation.pdf
2026-04

2026-04-27 19:11:43.053 | INFO     | pipelines.enterprise_knowledge_base.document_classification.pipeline:dlp_trigger:159 - Triggering DLP scan for: gs://ag-core-dev-fdx7-kb-landing-zone/SoW - Innovation.pdf
2026-04-27 19:11:43.054 | INFO     | pipelines.enterprise_knowledge_base.document_classification.dlp_service.service:inspect_gcs_file:38 - Starting DLP scan for: gs://ag-core-dev-fdx7-kb-landing-zone/SoW - Innovation.pdf
2026-04-27 19:11:43.054 | DEBUG    | pipelines.enterprise_knowledge_base.document_classification.dlp_service.service:_get_custom_keywords_config:209 - Building custom keywords config for inspection.
2026-04-27 19:11:43.777 | INFO     | pipelines.enterprise_knowledge_base.document_classification.dlp_service.service:inspect_gcs_file:66 - DLP Job created: projects/ag-core-dev-fdx7/locations/global/dlpJobs/i-6857005541962115804
2026-04-27 19:11:43.886 | INFO     | pipelines.enterprise_knowledge_base.document_classification.dlp_service.service:wait_for_job:104 - Waiting

In [ ]:
resp

OrchestratorRunResponse(gcs_uri='gs://kb-operations/orchestration-test/internal-use-only/emmanuel.amador/Carta_recomendación_CONUEE.pdf', chunks_generated=2, final_domain='operations', security_tier='internal-use-only')